# Consolidated basin list from `Caravan_all` and data availability checks

This notebook:

1. Builds a consolidated basin list across all subdatasets merged under `data/Caravan_all`.
2. Identifies GRDC basins that are duplicates of basins already present elsewhere in `Caravan_all` — either the original Caravan-core dataset (via the `nat_id` column in `attributes_additional_grdc.csv`) or one of the newer country extensions (via geographic proximity) — and drops the GRDC copies in favor of the original data.
3. Flags catchments with less than 10 years of valid (non-missing) streamflow data.

Outputs are written to this notebook's own folder (`notebooks/sb2026/`).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
DATA_DIR = Path("/home/ka/ka_iwu/ka_si3685/Hy2DL_dev-main/data/Caravan_all")
OUTPUT_DIR = Path("/home/ka/ka_iwu/ka_si3685/Hy2DL_dev-main/notebooks/sb2026")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_YEARS = 10  # threshold for flagging catchments with short streamflow records

SUBDATASET_CODES = sorted(d.name for d in (DATA_DIR / "attributes").iterdir() if d.is_dir())
SUBDATASET_CODES

## 1. Build the consolidated basin list

Every subdataset has an `attributes_other_<code>.csv` file with the columns `gauge_id`, `gauge_name`, `country`, `gauge_lat`, `gauge_lon`, `area`. We load and concatenate these across all subdataset codes found under `Caravan_all/attributes/`.

In [ ]:
def load_basin_metadata(code: str) -> pd.DataFrame:
    path = DATA_DIR / "attributes" / code / f"attributes_other_{code}.csv"
    df = pd.read_csv(path)
    df["subdataset"] = code
    return df[["gauge_id", "subdataset", "country", "gauge_lat", "gauge_lon", "area"]]


basin_list = pd.concat(
    [load_basin_metadata(code) for code in SUBDATASET_CODES], ignore_index=True
)
print(f"Total basins across all {len(SUBDATASET_CODES)} subdatasets: {len(basin_list)}")
basin_list.head()

## 2. Identify GRDC duplicates of Caravan-core and other Caravan extensions

### 2a. ID-based match against Caravan-core

`attributes_additional_grdc.csv` has a `nat_id` column holding each gauge's national/official identifier. The original 7 Caravan-core datasets (`camels`, `camelsaus`, `camelsbr`, `camelsgb`, `camelscl`, `hysets`, `lamah`) reuse these same national identifiers as the suffix of their `gauge_id` (e.g. `camels_01013500` → USGS site `01013500`; `lamah_200154` → Austrian HZB number `200154`).

We match GRDC's `nat_id` (restricted to the matching country) against the stripped IDs of these 7 core datasets to find physical-station duplicates.

### 2b. Geographic match against other Caravan extensions

The other community extensions merged into `Caravan_all` (DE, DK, CH, CZ, ES, IL, LamaH-Ice, AUS-VIC) are separate from the original Caravan core and use their **own** internal numbering schemes that don't line up with GRDC's `nat_id` at all — e.g. `camelsde_DE110000` has no relation to the raw German gauge number GRDC uses for the same station. ID-based matching finds zero overlap for these even where real physical duplicates exist (confirmed for Germany: 214/336 GRDC-DE stations turned out to be within 1 km of a `camelsde` station once checked geographically).

So for these 8 extensions we instead match by geographic proximity: for every GRDC station, find the nearest station in the corresponding extension (restricted to the same country) and flag it as a duplicate if the two are within 1 km of each other.

In [ ]:
CORE_DATASETS = ["camels", "camelsaus", "camelsbr", "camelsgb", "camelscl", "hysets", "lamah"]

# Map GRDC's 2-letter ISO country codes (used in attributes_additional_grdc.csv) to the
# full country names used in the core datasets' attributes_other_<code>.csv, and to which
# core dataset(s) may contain overlapping stations for that country. Order matters when a
# country maps to more than one dataset (e.g. US): the first match found wins.
GRDC_COUNTRY_TO_CORE = {
    "US": [("camels", "United States of America"), ("hysets", "United States of America")],
    "AU": [("camelsaus", "Australia")],
    "BR": [("camelsbr", "Brazil")],
    "CA": [("hysets", "Canada")],
    "AT": [("lamah", "Austria")],
    "CH": [("lamah", "Switzerland")],
    "DE": [("lamah", "Germany")],
    "CZ": [("lamah", "Czech Republic")],
    "GB": [
        ("camelsgb", "Great Britain"),
        ("camelsgb", "England"),
        ("camelsgb", "Scotland"),
        ("camelsgb", "Wales"),
    ],
    "CL": [("camelscl", "Chile")],
}


def stripped_id(gauge_id: str) -> str:
    return gauge_id.split("_", 1)[1].upper() if "_" in gauge_id else gauge_id.upper()


# (dataset_code, country_name) -> set of stripped national ids
core_id_lookup = {}
for code in CORE_DATASETS:
    df = load_basin_metadata(code)
    for country_name, group in df.groupby("country"):
        core_id_lookup[(code, country_name)] = set(group["gauge_id"].map(stripped_id))

In [ ]:
grdc_extra = pd.read_csv(
    DATA_DIR / "attributes" / "grdc" / "attributes_additional_grdc.csv",
    usecols=["gauge_id", "country", "nat_id"],
)
grdc_extra["nat_id"] = grdc_extra["nat_id"].astype(str).str.strip().str.upper()


def find_core_match(row):
    for code, country_name in GRDC_COUNTRY_TO_CORE.get(row["country"], []):
        idset = core_id_lookup.get((code, country_name))
        if idset and row["nat_id"] in idset:
            return code
    return None


grdc_extra["duplicate_of_core"] = grdc_extra.apply(find_core_match, axis=1)
duplicates_core = grdc_extra[grdc_extra["duplicate_of_core"].notna()].copy()

print(f"GRDC basins identified as duplicates of Caravan-core: {len(duplicates_core)} / {len(grdc_extra)}")
duplicates_core["duplicate_of_core"].value_counts()

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


EXTENSION_DATASETS = [
    "camelsde", "camelsdk", "camelsch", "camelscz", "camelses", "il", "lamahice", "ausvic",
]

# Some extensions label country in a way that differs from GRDC's full country names
# (e.g. camelsch labels stations "CH"/"A"/"DE"/"FR"/"I" for border-crossing catchments;
# camelscz uses "Czechia" where GRDC uses "Czech Republic"). Identity mapping otherwise.
COUNTRY_NAME_TRANSLATION = {
    "CH": "Switzerland",
    "A": "Austria",
    "DE": "Germany",
    "FR": "France",
    "I": "Italy",
    "Czechia": "Czech Republic",
}

DIST_THRESHOLD_KM = 1.0  # stations closer than this are treated as the same physical gauge

grdc_other = pd.read_csv(
    DATA_DIR / "attributes" / "grdc" / "attributes_other_grdc.csv",
    usecols=["gauge_id", "country", "gauge_lat", "gauge_lon", "area"],
)

geo_matches = []
for code in EXTENSION_DATASETS:
    ext = load_basin_metadata(code).copy()
    ext["country_norm"] = ext["country"].map(lambda c: COUNTRY_NAME_TRANSLATION.get(c, c))
    for country_norm, ext_group in ext.groupby("country_norm"):
        grdc_subset = grdc_other[grdc_other["country"] == country_norm]
        if grdc_subset.empty:
            continue
        ext_lat = ext_group["gauge_lat"].to_numpy()
        ext_lon = ext_group["gauge_lon"].to_numpy()
        ext_id = ext_group["gauge_id"].to_numpy()
        for grow in grdc_subset.itertuples(index=False):
            d = haversine_km(grow.gauge_lat, grow.gauge_lon, ext_lat, ext_lon)
            idx = np.argmin(d)
            if d[idx] < DIST_THRESHOLD_KM:
                geo_matches.append(
                    {
                        "gauge_id": grow.gauge_id,
                        "country": country_norm,
                        "duplicate_of_extension": code,
                        "matched_gauge_id": ext_id[idx],
                        "distance_km": d[idx],
                    }
                )

duplicates_geo = pd.DataFrame(geo_matches)
print(
    f"GRDC basins identified as duplicates of Caravan extensions "
    f"(geographic match, <{DIST_THRESHOLD_KM} km): {len(duplicates_geo)}"
)
duplicates_geo["duplicate_of_extension"].value_counts()

In [ ]:
duplicates_core_out = duplicates_core.rename(columns={"duplicate_of_core": "matched_dataset"}).copy()
duplicates_core_out["match_method"] = "id"
duplicates_core_out["matched_gauge_id"] = np.nan
duplicates_core_out["distance_km"] = np.nan

duplicates_geo_out = duplicates_geo.rename(columns={"duplicate_of_extension": "matched_dataset"}).copy()
duplicates_geo_out["match_method"] = "geo"
duplicates_geo_out["nat_id"] = np.nan

common_cols = ["gauge_id", "country", "match_method", "matched_dataset", "matched_gauge_id", "nat_id", "distance_km"]
all_duplicates = pd.concat(
    [duplicates_core_out[common_cols], duplicates_geo_out[common_cols]],
    ignore_index=True,
)

# A GRDC gauge could in principle be flagged by both methods; every match is kept in the
# saved file for transparency, but only one removal happens per unique GRDC gauge_id.
print(
    f"Total duplicate flags: {len(all_duplicates)}  |  "
    f"unique GRDC gauges flagged: {all_duplicates['gauge_id'].nunique()}"
)
all_duplicates["match_method"].value_counts()

In [ ]:
duplicates_path = OUTPUT_DIR / "grdc_duplicates.csv"
all_duplicates.to_csv(duplicates_path, index=False)
print(
    f"Saved GRDC duplicate mapping ({len(all_duplicates)} rows, "
    f"{all_duplicates['gauge_id'].nunique()} unique GRDC gauges) to {duplicates_path}"
)

# Drop the GRDC copies; the matching core/extension basin is already present in basin_list,
# so no rows need to be added back in — we just remove the redundant GRDC entries.
duplicate_grdc_ids = set(all_duplicates["gauge_id"])
basin_list_dedup = basin_list[~basin_list["gauge_id"].isin(duplicate_grdc_ids)].reset_index(drop=True)

print(f"Basins before dedup: {len(basin_list)}  |  after removing GRDC duplicates: {len(basin_list_dedup)}")

## 3. Save the consolidated (deduplicated) basin list

In [ ]:
basin_list_path = OUTPUT_DIR / "consolidated_basin_list.csv"
basin_list_dedup.to_csv(basin_list_path, index=False)
print(f"Saved consolidated basin list ({len(basin_list_dedup)} basins) to {basin_list_path}")
basin_list_dedup["subdataset"].value_counts()

## 4. Data availability check — catchments with less than 10 years of streamflow data

For every basin in the deduplicated list we read its timeseries CSV (only the `date` and `streamflow` columns, for speed) and compute:

- `valid_years_streamflow`: number of non-missing daily streamflow observations divided by 365.25. This is the number of years of *usable* data.
- `record_span_years`: the calendar span between the first and last valid streamflow observation. Useful to distinguish a genuinely short record from a long-but-gappy one.

This scans one file per basin (tens of thousands of files) and may take a few minutes.

In [ ]:
def timeseries_path(gauge_id: str, subdataset: str) -> Path:
    return DATA_DIR / "timeseries" / "csv" / subdataset / f"{gauge_id}.csv"


def streamflow_availability(path: Path) -> tuple[float, float]:
    df = pd.read_csv(path, usecols=["date", "streamflow"], parse_dates=["date"])
    valid = df.loc[df["streamflow"].notna(), "date"]
    valid_years = len(valid) / 365.25
    span_years = (valid.max() - valid.min()).days / 365.25 if len(valid) else 0.0
    return valid_years, span_years


records = []
total = len(basin_list_dedup)
for i, row in enumerate(basin_list_dedup.itertuples(index=False), start=1):
    path = timeseries_path(row.gauge_id, row.subdataset)
    try:
        valid_years, span_years = streamflow_availability(path)
    except FileNotFoundError:
        valid_years, span_years = np.nan, np.nan
    records.append(
        {
            "gauge_id": row.gauge_id,
            "subdataset": row.subdataset,
            "valid_years_streamflow": valid_years,
            "record_span_years": span_years,
        }
    )
    if i % 2000 == 0 or i == total:
        print(f"Processed {i}/{total} basins...")

availability = pd.DataFrame(records)

In [ ]:
basin_list_full = basin_list_dedup.merge(availability, on=["gauge_id", "subdataset"], how="left")

short_record_basins = basin_list_full[basin_list_full["valid_years_streamflow"] < MIN_YEARS].copy()
short_record_basins = short_record_basins.sort_values("valid_years_streamflow")

print(
    f"Catchments with less than {MIN_YEARS} years of valid streamflow data: "
    f"{len(short_record_basins)} / {len(basin_list_full)}"
)
short_record_basins["subdataset"].value_counts()

In [ ]:
short_record_path = OUTPUT_DIR / "basins_less_than_10yr.csv"
short_record_basins.to_csv(short_record_path, index=False)
print(f"Saved short-record basin list ({len(short_record_basins)} rows) to {short_record_path}")

basin_list_full_path = OUTPUT_DIR / "consolidated_basin_list_with_availability.csv"
basin_list_full.to_csv(basin_list_full_path, index=False)
print(f"Saved full consolidated basin list with data-availability columns to {basin_list_full_path}")

## Summary of outputs

All written to `notebooks/sb2026/`:

- `grdc_duplicates.csv` — GRDC basins identified as duplicates, whether by ID match against Caravan-core (`match_method="id"`) or geographic proximity against another Caravan extension (`match_method="geo"`), with the matched dataset/gauge and (for geo matches) the distance in km.
- `consolidated_basin_list.csv` — deduplicated basin metadata (`gauge_id`, `subdataset`, `country`, `gauge_lat`, `gauge_lon`, `area`) across all of `Caravan_all`.
- `consolidated_basin_list_with_availability.csv` — the same list with `valid_years_streamflow` and `record_span_years` added.
- `basins_less_than_10yr.csv` — subset of catchments with fewer than `MIN_YEARS` (10) years of valid streamflow data, sorted ascending.